# MixIT-MSS - Separation & Listening

Notebook to run inference with a trained MixIT / BS-Locoformer model and listen to
(or save) the separated tracks.

It covers **both scenarios**:
- **RAW** - the pre-training checkpoint (`pretrained_mixit.pth`): emits the *N* (e.g. 12)
  unlabeled MixIT channels. Diagnostic: hear *what the unsupervised model learned*.
- **FINE-TUNED** - the fine-tuning checkpoint (`finetuned_musdb.pth`): emits the 4 VDBO
  stems (vocals / drums / bass / other), using the saved channel map.

Target data: the **MUSDB18 test split**.

## 1. Setup & config

In [2]:
# If you did NOT `pip install -e .`, uncomment to add the project root to the path:
# import sys; sys.path.insert(0, "/nas/home/macerbi/mixit-mss")

import os
import torch
import torchaudio
from IPython.display import Audio, display

from mixit_mss.inference import (
    load_separator, separate, separate_waveform,
    separate_musdb_test, save_stems,
)
from mixit_mss.evaluation import (
    evaluate_musdb_test,
    evaluate_pretrained_with_selection,
    summarize
)

def _pick_free_gpu() -> torch.device:
    # Return the CUDA device with the most free memory; fall back to CPU.
    if not torch.cuda.is_available():
        return torch.device("cpu")
    best_idx, best_free = 0, -1
    for i in range(torch.cuda.device_count()):
        try:
            free, _ = torch.cuda.mem_get_info(i)
        except Exception:
            free = 0  # GPU fully occupied or unavailable
        if free > best_free:
            best_free, best_idx = free, i
    if best_free <= 0:
        print("Warning: all GPUs are fully occupied. Using cuda:0 anyway.")
    return torch.device(f"cuda:{best_idx}")

DEVICE = _pick_free_gpu()

print(f"Device   : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(DEVICE)}")
    print(f"VRAM: {torch.cuda.get_device_properties(DEVICE).total_memory / 1e9:.1f} GB")

Device   : cuda:0
Device name: NVIDIA TITAN RTX
VRAM: 25.2 GB


In [3]:
# Checkpoints (edit to your files)
PRETRAIN_CKPT  = "/nas/home/macerbi/mixit-mss/pretrained_mixit.pth"     # RAW scenario
FINETUNE_CKPT  = "/nas/home/macerbi/mixit-mss/finetuned_musdb.pth"      # FINE-TUNED scenario (may not exist yet)

# MUSDB18 train split root (standard layout: <root>/<track>/mixture.wav)
MUSDB_TRAIN_ROOT = "/nas/home/macerbi/dataset/musdb18hq/train"

# MUSDB18 test split root (standard layout: <root>/<track>/mixture.wav)
MUSDB_TEST_ROOT = "/nas/home/macerbi/dataset/musdb18hq/test"

# Where to write separated wavs
OUT_ROOT = "/nas/home/macerbi/mixit-mss/runs"

# IMPORTANT: these MUST match the values used at TRAINING time. If you trained with
# reduced settings for memory (e.g. --n_layers 4), change them here too, otherwise
# the weights won't load correctly.
MODEL_KW = dict(
    n_srcs=12,          # --n_srcs
    n_channels=2,       # stereo
    n_layers=6,         # --n_layers
    emb_dim=128,        # --emb_dim
    sr=44100,           # --sr
    stft_size=2048,     # --stft_size
    hop_length=512,     # --hop_length
)

# Inference (overlap-add) settings
CHUNK_SECONDS = 6.0     # window length; lower if you hit OOM at inference
OVERLAP = 0.5           # 50% overlap between chunks

## 2. Load a separator

`load_separator` auto-detects the scenario from the checkpoint contents:
- a plain state_dict $\rightarrow$ RAW (pre-trained)
- a dict with a `channel_map` $\rightarrow$ FINE-TUNED.

In [4]:
# Pick ONE checkpoint to load. Start with whichever you have trained.
CKPT = PRETRAIN_CKPT          # or: CKPT = FINETUNE_CKPT

sep = load_separator(CKPT, device=DEVICE, **MODEL_KW)
print("loaded:", CKPT)
print("fine-tuned (VDBO stems)?", sep.is_finetuned)
if sep.is_finetuned:
    print("channel map (stem -> channel):", sep.channel_map)
else:
    print(f"RAW mode: model emits {sep.n_srcs} unlabeled channels")

Band-split module has 62 bands


/nas/home/macerbi/miniconda3/envs/msslnet/lib/python3.10/site-packages/rotary_embedding_torch/rotary_embedding_torch.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/nas/home/macerbi/miniconda3/envs/msslnet/lib/python3.10/site-packages/rotary_embedding_torch/rotary_embedding_torch.py:254: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/nas/home/macerbi/mixit-mss/mixit_mss/bslocoformer/tflocoformer_separator.py:521: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


loaded: /nas/home/macerbi/mixit-mss/pretrained_mixit.pth
fine-tuned (VDBO stems)? False
RAW mode: model emits 12 unlabeled channels


## 3. Separate a single MUSDB test track and listen

We take one track's `mixture.wav`, separate it, and play the results inline.

- In **FINE-TUNED** mode you get 4 named stems.
- In **RAW** mode you get N channels named `src00..srcNN` - expect some
  over-separation (an instrument split across channels); this is diagnostic.

In [ ]:
# List available test tracks
tracks = sorted(d for d in os.listdir(MUSDB_TEST_ROOT)
                if os.path.isdir(os.path.join(MUSDB_TEST_ROOT, d)))
print(f"{len(tracks)} test tracks. First few:")
for t in tracks[:5]:
    print("  ", t)

In [ ]:
# Choose a track and separate it
track = tracks[1]
mix_path = os.path.join(MUSDB_TEST_ROOT, track, "mixture.wav")
print("separating:", track)

stems = separate(sep, mix_path, chunk_seconds=CHUNK_SECONDS, overlap=OVERLAP)
print("got", len(stems), "outputs:", list(stems.keys()))

In [ ]:
# Listen to the original mixture first (downmixed to mono just for the player)
mix, sr = torchaudio.load(mix_path)
print("MIXTURE")
display(Audio(mix.mean(0).numpy(), rate=sr))

In [ ]:
# Listen to each separated output.
# (stems[name] is a [C, L] tensor; we downmix to mono for the inline player.)
for name, wav in stems.items():
    print(name.upper())
    display(Audio(wav.mean(0).numpy(), rate=sep.sr))

### RAW mode helper - find which channels are "loud"

In RAW mode with 12 channels, many may be near-silent. This ranks channels by
energy so you can focus on the active ones.

In [ ]:
if not sep.is_finetuned:
    energies = {name: float((wav ** 2).mean()) for name, wav in stems.items()}
    ranked = sorted(energies.items(), key=lambda kv: kv[1], reverse=True)
    print("channels by energy (loudest first):")
    for name, e in ranked:
        print(f"  {name}: {e:.6f}")
    # play the top 4 loudest channels
    print("\nTop-4 loudest channels:")
    for name, _ in ranked[:4]:
        print(name.upper())
        display(Audio(stems[name].mean(0).numpy(), rate=sep.sr))
else:
    print("Fine-tuned model: stems are already the 4 VDBO sources.")

## 4. Save the separated tracks to disk

In [ ]:
# Write this track's stems to OUT_ROOT/<track>/<name>.wav
out_dir = os.path.join(OUT_ROOT, track)
paths = save_stems(stems, out_dir, sr=sep.sr)
print("written:")
for name, p in paths.items():
    print("  ", p)

## 5. Batch: separate the whole MUSDB test split

`separate_musdb_test` runs over every track and writes `OUT_ROOT/<track>/<stem>.wav`.
Use `limit` to do only the first few while experimenting; remove it for the full split.

In [ ]:
results = separate_musdb_test(
    sep, MUSDB_TEST_ROOT, OUT_ROOT,
    limit=3,                       # set to None for all test tracks
    chunk_seconds=CHUNK_SECONDS, overlap=OVERLAP,
)
print("\ndone:", len(results), "tracks")
for track, out_dir in results:
    print("  ", track, "->", out_dir)

## 6. Evaluate against the MUSDB ground truth (SDR)

Listening is qualitative; to get the numbers reported in the paper we score the
separated stems against the MUSDB ground truth. Two metrics (Saijo & Bando, Sec. 4.3):

- **uSDR** - song-level SDR (MDX convention). No extra dependency.
- **cSDR** - chunk-wise BSSEval-v4 SDR, median over 1 s frames. Needs `pip install museval`.

Requires a **fine-tuned (VDBO)** model: a raw pre-training model has unlabeled
channels that cannot be matched to the four stems.

In [ ]:
# The model must be fine-tuned (VDBO). If you loaded the pre-training checkpoint
# above, load the fine-tuned one instead before evaluating.
assert sep.is_finetuned, "Load a fine-tuned checkpoint (with a channel_map) to evaluate."

# metrics=("usdr",) is dependency-free; add "csdr" once museval is installed.
df = evaluate_musdb_test(
    sep, MUSDB_TEST_ROOT,
    metrics=("usdr",),          # e.g. ("usdr", "csdr")
    limit=5,                     # None = full test split (50 tracks)
    chunk_seconds=12.0, overlap=0.5,
)
df.head()

In [ ]:
# Per-stem averages plus the overall mean, in the style of the paper's Table 1.
summary = summarize(df)
summary.round(2)

### Compare pre-training vs from-scratch

The paper's central claim is that MixIT pre-training beats training from scratch.
To reproduce it, fine-tune **twice** - once with `--pretrained pretrained_mixit.pth`
and once without (random init) - then evaluate both and compare the `Average` row.
The gap is the value added by MixIT.

In [ ]:
# Example pattern (uncomment once you have both checkpoints):
# sep_pre   = load_separator("finetuned_from_pretrain.pth", device=device, **MODEL_KW)
# sep_scratch = load_separator("finetuned_from_scratch.pth", device=device, **MODEL_KW)
# df_pre    = evaluate_musdb_test(sep_pre,   MUSDB_TEST_ROOT, metrics=("usdr","csdr"))
# df_scratch= evaluate_musdb_test(sep_scratch, MUSDB_TEST_ROOT, metrics=("usdr","csdr"))
# print("pre-train:"); print(summarize(df_pre).round(2))
# print("scratch:");   print(summarize(df_scratch).round(2))

## 6b. Diagnostic: pre-trained model, channel selection only (no fine-tuning)

This reproduces the paper's channel-selection procedure (Sec. 4.3, steps 1-3) and
scores the **pre-training-only** model on the MUSDB test set - *without* any
fine-tuning. It measures what MixIT learned on its own, before supervision.

How it works:
1. load the **pre-training** checkpoint (12 unlabeled channels);
2. on the MUSDB *validation* tracks, find, per stem, the output channel most often
   aligned with it (max-SNR permutation) - this is `select_channels`;
3. evaluate the pre-trained model on the *test* tracks using that map, with the
   same uSDR / cSDR metrics used above.

Selection uses the train split and scoring uses the test split, so the map is not
tuned on the data it is scored on. Expect lower numbers than the fine-tuned model
(and than the paper's Table 2, which additionally fine-tunes encoder+decoder) - the
point is the gap, which shows how much separation MixIT already provides for free.

In [5]:
# Load the PRE-TRAINING checkpoint (not the fine-tuned one) for this diagnostic.
sep_pre = load_separator(PRETRAIN_CKPT, device=DEVICE, **MODEL_KW)
assert not sep_pre.is_finetuned, "Use the pre-training checkpoint here (no channel_map)."

df_pre, cmap = evaluate_pretrained_with_selection(
    sep_pre,
    musdb_train_root=MUSDB_TRAIN_ROOT,   # channel selection here
    musdb_test_root=MUSDB_TEST_ROOT,     # scoring here
    n_srcs=MODEL_KW["n_srcs"],
    metrics=("usdr",),                    # add "csdr" once museval is installed
    limit=5,                              # None = full test split
    sel_max_batches=8,                    # more batches = steadier channel map
)
print("selected channel map:", cmap)
summarize(df_pre).round(2)

Band-split module has 62 bands
running channel selection on the validation split...


/nas/home/macerbi/miniconda3/envs/msslnet/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


stem->channel map: {'bass': 7, 'vocals': 0, 'drums': 3, 'other': 5}
[1/5] AM Contra - Heart Peripheral | uSDR 0.35
[2/5] Al James - Schoolboy Facination | uSDR 1.80
[3/5] Angels In Amplifiers - I'm Alright | uSDR 1.52
[4/5] Arise - Run Run Run | uSDR 0.63
[5/5] BKS - Bulldozer | uSDR 0.01
selected channel map: {'bass': 7, 'vocals': 0, 'drums': 3, 'other': 5}


,uSDR
vocals,1.40
drums,0.80
bass,1.87
other,-0.62
Average,0.86


## 7. (Optional) Separate an arbitrary audio file

Not limited to MUSDB - point it at any stereo file.

In [ ]:
# my_file = "/path/to/any/song.wav"
# stems = separate(sep, my_file, chunk_seconds=CHUNK_SECONDS, overlap=OVERLAP)
# for name, wav in stems.items():
#     print(name.upper()); display(Audio(wav.mean(0).numpy(), rate=sep.sr))
# save_stems(stems, "sep_out/my_file", sr=sep.sr)